In [0]:
from pyspark.sql import functions as F

In [0]:
order_enriched = spark.read.table("ecommerce_catalog.silver.orders")
customer_enriched = spark.read.table("ecommerce_catalog.silver.customer")
product_enriched = spark.read.table("ecommerce_catalog.silver.products")
enriched_order_customer_product = spark.read.table("ecommerce_catalog.silver.enriched_order_customer_product")

In [0]:
def aggregate_by_profit(df):
  return df.groupBy(F.year(F.col("Order Date")).alias("Year"),
                    F.col("Product Category"),F.col("Product Sub-Category"),F.col("Customer Name")).agg(F.round(F.sum(F.col("Profit")),2).alias("Total Profit"))

In [0]:
df_aggregate_by_profit = aggregate_by_profit(enriched_order_customer_product )
df_aggregate_by_profit.write.mode("overwrite").option("delta.columnMapping.mode","name").saveAsTable("ecommerce_catalog.gold.profit_by_year_category_subcategory_customer")

In [0]:
# Using SQL output the following aggregates
# Profit by Year
# Profit by Year + Product Category
# Profit by Customer
# Profit by Customer + Year


In [0]:
%sql
select  Year,round(sum(`Total Profit`),2) yearly_profit from ecommerce_catalog.gold.profit_by_year_category_subcategory_customer
group by Year 

In [0]:
%sql
select  Year,`Product Category`,round(sum(`Total Profit`),2) yearly_profit_by_product_category from ecommerce_catalog.gold.profit_by_year_category_subcategory_customer
group by Year ,`Product Category`

In [0]:
%sql
select  Year,`Customer Name`,round(sum(`Total Profit`),2) yearly_profit_by_customer from ecommerce_catalog.gold.profit_by_year_category_subcategory_customer
group by Year ,`Customer Name`

In [0]:
%sql
select  `Customer Name`,round(sum(`Total Profit`),2) profit_by_customer from ecommerce_catalog.gold.profit_by_year_category_subcategory_customer
group by `Customer Name`